# BPIC-17

## Step 1: Setup & Data Loading

In [1]:
import pm4py
import pandas as pd
import numpy as np
from IPython.display import display

CASE_COL = "case:concept:name"
ACT_COL  = "concept:name"
TS_COL   = "time:timestamp"

log_raw = pm4py.read_xes("BPIC2017.xes")
log_raw = pm4py.format_dataframe(
    log_raw,
    case_id=CASE_COL,
    activity_key=ACT_COL,
    timestamp_key=TS_COL,
)
print(f"log_raw: {log_raw[CASE_COL].nunique():,} cases  |  {len(log_raw):,} events")

/Users/cole/.local/share/virtualenvs/BPM-lrkv0Mze/lib/python3.14/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

log_raw: 31,509 cases  |  1,202,267 events


## Step 2: Simple Event Log Analysis (§ 3.2)

In [ ]:
def compute_variants(df):
    seqs = df.groupby('case:concept:name')['concept:name'].apply(tuple)
    return seqs.nunique()

def case_length_stats(df):
    lengths = df.groupby('case:concept:name').size()
    return lengths.mean(), lengths.std()

def case_duration_stats(df):
    dur = df.groupby('case:concept:name')['time:timestamp'].agg(lambda x: x.max() - x.min())
    dur_days = dur.dt.total_seconds() / 86400
    dur_min  = dur.dt.total_seconds() / 60
    return dur_days.mean(), dur_days.std(), dur_min.mean(), dur_min.std()

def more_than_one_offer(df):
    offer_counts = (
        df[df['concept:name'] == 'O_Create Offer']
        .groupby('case:concept:name').size()
    )
    total_cases = df['case:concept:name'].nunique()
    return (offer_counts > 1).sum() / total_cases

def acceptance_rate(df):
    cases_with_pending = df[df['concept:name'] == 'A_Pending']['case:concept:name'].nunique()
    total_cases = df['case:concept:name'].nunique()
    return cases_with_pending / total_cases

In [ ]:
n_cases    = log_raw['case:concept:name'].nunique()
n_events   = len(log_raw)
n_variants = compute_variants(log_raw)

case_cols   = [c for c in log_raw.columns if c.startswith('case:')]
n_distinct_case_attrs = len(case_cols)

n_distinct_labels = log_raw['concept:name'].nunique()

cat_cols = [c for c in log_raw.columns if log_raw[c].dtype == object and c != 'case:concept:name']

len_mean, len_std = case_length_stats(log_raw)
dur_d_mean, dur_d_std, dur_m_mean, dur_m_std = case_duration_stats(log_raw)

more_than_one_offer_cases = more_than_one_offer(log_raw)
acc = acceptance_rate(log_raw)

print('Metrics computed.')

Metrics computed.


In [ ]:
metrics = [
    ('Number of cases',              f'{n_cases:,}'),
    ('Number of events',             f'{n_events:,}'),
    ('Number of variants',           f'{n_variants:,}'),
    ('Distinct case attributes',     f'{n_distinct_case_attrs}  {case_cols}'),
    ('Distinct event labels',        f'{n_distinct_labels}'),
    ('Categorical event attributes', f'{len(cat_cols)}  {cat_cols}'),
    ('Case length mean ± SD',        f'{len_mean:.1f} ± {len_std:.1f} events'),
    ('Case duration mean ± SD',      f'{dur_d_mean:.1f} ± {dur_d_std:.1f} days  '
                                     f'({dur_m_mean:.0f} ± {dur_m_std:.0f} min)'),
    ('Cases with >1 offer (custom 1)', f'{more_than_one_offer_cases:.3f}  ({more_than_one_offer_cases*100:.1f}% of cases received more than one offer)'),
    ('Loan grant rate (custom 2)',   f'{acc:.3f}  ({acc*100:.1f}% of cases contain A_Pending)'),
]

summary = pd.DataFrame(metrics, columns=['Metric', 'Value'])
summary

,Metric,Value
0,Number of cases,"31,509"
1,Number of events,"1,202,267"
2,Number of variants,"15,930"
3,Distinct case attributes,"4 ['case:LoanGoal', 'case:ApplicationType', '..."
4,Distinct event labels,26
5,Categorical event attributes,"2 ['Accepted', 'Selected']"
6,Case length mean ± SD,38.2 ± 16.7 events
7,Case duration mean ± SD,21.9 ± 13.2 days (31535 ± 18964 min)
8,Cases with >1 offer (custom 1),0.272 (27.2% of cases received more than one ...
9,Loan grant rate (custom 2),0.547 (54.7% of cases contain A_Pending)
